In [1]:
import os

In [2]:
%pwd

'c:\\Users\\hulkh\\Desktop\\Python\\supplier_clustering\\Supplier_clustering\\research'

In [3]:
os.chdir("C:/Users/hulkh/Desktop/Python/supplier_clustering/Supplier_clustering")

In [4]:
#entity
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.Supplier_clustering.constants import *
from src.Supplier_clustering.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
            config = self.config.data_ingestion
    
            create_directories([config.root_dir])
    
            data_ingestion_config = DataIngestionConfig(
                root_dir=config.root_dir,
                source_URL=config.source_URL,
                local_data_file=config.local_data_file,
                unzip_dir=config.unzip_dir 
            )
    
            return data_ingestion_config

In [8]:
# components
import os
import urllib.request as request
import zipfile
from Supplier_clustering.utils.logger import logger
from Supplier_clustering.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-08-20 21:02:56,496: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-20 21:02:56,498: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-20 21:02:56,500: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-08-20 21:02:56,502: INFO: common: created directory at: artifacts]
[2026-08-20 21:02:56,503: INFO: common: created directory at: artifacts/data_ingestion]
[2026-08-20 21:02:57,453: INFO: 2280763589: artifacts/data_ingestion/data.zip download! with following info: 
Date: Thu, 20 Aug 2026 17:02:57 GMT
Content-Type: text/html; charset=utf-8
x-repository-download: git clone https://github.com/Kiran-Samuel/Supplier_clustering.git
x-raw-download: https://raw.githubusercontent.com/Kiran-Samuel/Supplier_clustering/main/purchase_orders.zip
Vary: X-PJAX, X-PJAX-Container, Turbo-Visit, Turbo-Frame, X-Requested-With, X-GitHub-Client-Version, Sec-Fetch-Site,Accept-Encoding, Accept, X-Requested-With
ETag: W/"ad4273f3be1522ca81fca937307495a

BadZipFile: File is not a zip file